In [1]:
import torch
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from seqeval.scheme import IOB2
from transformers import AutoTokenizer
import pandas as pd
import os
from transformers import TrainingArguments, Trainer
from datasets import Dataset
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'device used: PC GPU - {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"device used: MAC GPU - MPS")
else:
    device = torch.device("cpu")
    print(f"CPU used")

device used: PC GPU - NVIDIA GeForce RTX 5070 Ti


In [3]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

def align_labels_tokens(labels, word_ids):
    new_labels = []
    current_word = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)

        elif word_id != current_word:
            current_word = word_id
            new_labels.append(labels[word_id])

        else:
            new_labels.append(-100)

    return new_labels

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

c:\Users\dluc4\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dluc4\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
sample_tokens = ['Experience','with', 'micro-services','like','Kubernetes']
sample_tags = ['O','O','B-KNOWLEDGE','O','B-KNOWLEDGE']

tokenize_input = tokenizer(sample_tokens, is_split_into_words=True)
word_ids = tokenize_input.word_ids()

print(tokenize_input)
print(word_ids)

aligned_labels = align_labels_tokens(sample_tags, word_ids)

subwords = tokenizer.convert_ids_to_tokens(tokenize_input["input_ids"])

print(f"{'SUBWORD':<15} | {'ORIGINAL WORD ID':<18} | {'ALIGNED LABEL'}")
print("------------------------------------------------------------------")
for subword, word_id, label in zip(subwords, word_ids, aligned_labels):
    print(f"{'subword':<15} | {str(word_id):<18} | {label}")

{'input_ids': [0, 14360, 19, 5177, 12, 28023, 101, 9609, 1943, 4135, 293, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[None, 0, 1, 2, 2, 2, 3, 4, 4, 4, 4, None]
SUBWORD         | ORIGINAL WORD ID   | ALIGNED LABEL
------------------------------------------------------------------
subword         | None               | -100
subword         | 0                  | O
subword         | 1                  | O
subword         | 2                  | B-KNOWLEDGE
subword         | 2                  | -100
subword         | 2                  | -100
subword         | 3                  | O
subword         | 4                  | B-KNOWLEDGE
subword         | 4                  | -100
subword         | 4                  | -100
subword         | 4                  | -100
subword         | None               | -100


In [5]:
df = pd.read_parquet("../data/df_cleaned.parquet")
df.head()

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,tokens_length
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train,0,0,False,7
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train,0,0,False,5
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train,0,0,False,3
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train,0,0,False,6
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train,0,0,False,2


In [6]:
unique_skills = sorted(set(tag for row in df['tags_skill'] for tag in row))
unique_knowledge = sorted(set(tag for row in df['tags_knowledge'] for tag in row))

skills2id = {tag: i for i, tag in enumerate(unique_skills)}
knowledge2id = {tag: i for i, tag in enumerate(unique_knowledge)}

id2skills = {i: tag for tag,i in skills2id.items()}
id2knowledge = {i: tag for tag,i in knowledge2id.items()}

print(skills2id)
print(knowledge2id)


{'B': 0, 'I': 1, 'O': 2}
{'B': 0, 'I': 1, 'O': 2}


In [7]:
def tokenize_align(sample, label_col, label2id):
    tokenize_inputs = tokenizer(
        sample["tokens"],
        truncation = True,
        max_length = 512,
        is_split_into_words = True,
        return_overflowing_tokens = True,
        stride = 50
    )

    labels = []

    for i, seq_idx in enumerate(tokenize_inputs.pop("overflow_to_sample_mapping")):     # map overflowing chunks back
        word_ids = tokenize_inputs.word_ids(batch_index = i)
        org_labels = sample[label_col][seq_idx]

        label_ids = [label2id[label] for label in org_labels]       # convert string labels to IDs

        aligned = align_labels_tokens(label_ids,word_ids)
        labels.append(aligned)

    tokenize_inputs["labels"] = labels
    return tokenize_inputs

In [8]:
def get_compute_metrics(id2label_dict):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [id2label_dict[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [id2label_dict[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        return {
            "Precision": precision_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
            "Recall": recall_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
            "F1_Score": f1_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
        }
    return compute_metrics

In [9]:
def train_and_eval(model_name, entity, label_col, labels2id, id2labels, X_train, X_val, X_test):
    print(f"Initialize Training Model: {model_name} for: {entity}")

    columns_to_remove = X_train.column_names
    train_token = X_train.map(lambda x: tokenize_align(x, label_col, labels2id), batched = True, remove_columns = columns_to_remove)
    val_token = X_val.map(lambda x: tokenize_align(x, label_col, labels2id), batched = True, remove_columns = columns_to_remove)
    test_token = X_test.map(lambda x: tokenize_align(x, label_col, labels2id), batched = True, remove_columns = columns_to_remove)

    model = AutoModelForTokenClassification.from_pretrained(
        model_name, 
        num_labels = len(id2labels),
        id2label = id2labels,
        label2id = labels2id
    ).to(device)

    data_collator = DataCollatorForTokenClassification(tokenizer)

    new_model_name = model_name.split('/')[-1]

    entity_training_args = TrainingArguments(
        output_dir= f"../models/{new_model_name}_{entity}",
        eval_strategy = "epoch",
        save_strategy= "epoch",
        load_best_model_at_end= True,
        metric_for_best_model= "F1_Score",
        greater_is_better= True,
        learning_rate= 1e-05,
        per_device_eval_batch_size= 16,
        per_device_train_batch_size=16,
        num_train_epochs= 3,
        weight_decay= 0.01,
        logging_steps=10
    )

    trainer = Trainer(
        model = model,
        args = entity_training_args,
        train_dataset=train_token,
        eval_dataset= val_token,
        data_collator= data_collator,
        processing_class = tokenizer,
        compute_metrics= get_compute_metrics(id2labels)
    )

    trainer.train()

    y_pred = trainer.predict(test_token)

    results = get_compute_metrics(id2labels)((y_pred.predictions, y_pred.label_ids))

    results_df = pd.DataFrame([{
        "Model": model_name,
        "Entity": entity,
        "Precision": round(results["Precision"], 4),
        "Recall": round(results["Recall"], 4),
        "F1_Score": round(results["F1_Score"], 4),
    }])

    os.makedirs("../results", exist_ok= True)
    file_path = f"../results/{new_model_name}_{entity}_results.csv"
    results_df.to_csv(file_path, index= False)

    print("Training and Evaluation complete!")
    print(f"Testing Metrics Saved to {file_path}")

    display(results_df)

    return results

In [10]:
X_train = Dataset.from_pandas(df[df['split'] == "train"])
X_val = Dataset.from_pandas(df[df['split'] == "validation"])
X_test = Dataset.from_pandas(df[df['split'] == "test"])

---

In [11]:
roberta_skill_metrics = train_and_eval(model_name= "roberta-base", entity="Skill", label_col= "tags_skill", labels2id= skills2id, id2labels= id2skills, X_train=X_train, X_val=X_val, X_test= X_test)

Initialize Training Model: roberta-base for: Skill


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training and Evaluation complete!
Testing Metrics Saved to ../results/roberta-base_Skill_results.csv


,Model,Entity,Precision,Recall,F1_Score
0,roberta-base,Skill,0.4862,0.4675,0.4766


In [12]:
roberta_knowledge_metrics = train_and_eval(model_name= "roberta-base", entity="Knowledge", label_col= "tags_knowledge", labels2id= knowledge2id, id2labels= id2knowledge, X_train=X_train, X_val=X_val, X_test= X_test)

Initialize Training Model: roberta-base for: Knowledge


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training and Evaluation complete!
Testing Metrics Saved to ../results/roberta-base_Knowledge_results.csv


,Model,Entity,Precision,Recall,F1_Score
0,roberta-base,Knowledge,0.5915,0.6908,0.6373
